# AELIONIX BLACKFORGE — Phase 12 Colab Validation

This notebook performs a deterministic, one-click validation of the **Container &
Kubernetes Security Capability Foundation** (Phase 12).

It exercises the full `blackforge.container` pipeline on a synthetic Kubernetes
**management-plane estate** — `aelionix-platform` (3 namespaces, 6 workloads /
deployments, 4 pods, 4 containers, 3 images behind 1 registry, services, an
ingress, RBAC roles/permissions, service accounts, network policies, security
contexts, resource limits):

* **cluster / node / namespace observation** — the estate topology root
* **workload & deployment observation** — one source row typed into both
  `workload` *and* `deployment` observations (`deploys` edges)
* **pod / container / image & registry observation** — registries are emitted
  *before* the images that belong to them so the `BELONGS_TO` edge resolves;
  containers link to images through `USES_IMAGE`
* **service / ingress / network policy / RBAC / service-account observation**
  — structural links only (`SELECTS`, `ROUTES_TO`, `APPLIES_TO`,
  `HAS_ROLE`, `HAS_PERMISSION`, `USES_SERVICE_ACCOUNT`)
* **security-context & resource-configuration observation** — container
  hardening assertions (`privileged`, `allow_privilege_escalation`,
  `run_as_non_root`, ...) and per-workload CPU/memory limits with declared-vs-
  cluster-reported **configuration discrepancies** that surface at
  `INFERRED` confidence instead of silently overwriting
* **credential redaction** — registry / service-account tokens are stripped
  while structural rows persist
* **guarded pipeline** — request validation, scope / authorization, target
  resolution, **fail-closed capability validation**, mock transport (no real
  cluster is ever queried or mutated), normalization, evidence persistence,
  world-model materialization, and best-effort memory linking

Every capability is risk **LOW**, mode **PASSIVE**, and supported on
`ASSET` / `CLOUD` targets. No exploitation, no attack-graph vocabulary, no
generic command execution surface.

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.

---

In [ ]:
import sys
import platform

print("Blackforge Phase 12 Colab Validation (Container & Kubernetes Security Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.evidence.repository",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.container",
    "blackforge.container.models",
    "blackforge.container.capabilities",
    "blackforge.container.transport",
    "blackforge.container.redaction",
    "blackforge.container.evidence",
    "blackforge.container.normalization",
    "blackforge.container.materializer",
    "blackforge.container.engine",
    "blackforge.container.addressing",
    "blackforge.container.canonical",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Container module imports: PASS")

---

In [ ]:
import subprocess
import sys
import os

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
# The subprocess runs against pristine defaults: BLACKFORGE_* env overrides
# from the runtime are stripped so the suite behaves exactly like CI and no
# ambient config skews assertions (e.g. log level / DB path).
_test_env = {k: v for k, v in os.environ.items() if not k.startswith("BLACKFORGE_")}
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR), env=_test_env,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase12_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "network_ready", "identity_ready",
            "authorization_ready", "model_router_ready", "cloud_ready",
            "container_ready"):
    assert verification[key], f"{key} must be True"
assert verification["container_ready"] is True, "container_ready must be True (14 typed capabilities)"
assert len(app.capability_registry.list_capabilities()) == 95

BOOTSTRAP_OK = app.healthy() and bool(verification["container_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (container_ready, 95 registered capabilities): PASS")

---

In [ ]:
from blackforge.container import (
    CONTAINER_CAPABILITY_IDS,
    ContainerMode,
    ContainerRequest,
)
from blackforge.scope.models import TargetScope, Target
from blackforge.core.types import RiskLevel, TargetType

MID = "mission_phase12_container"

PLATFORM = "aelionix-platform"
STAGING = "aelionix-staging"
FRONTEND = "aelionix-platform/frontend"
_ERROR_TARGETS = [
    "snail-cluster",
    "bursty-cluster",
    "locked-cluster",
    "garbled-cluster",
    "fabricated-cluster",
]
ALL = [PLATFORM, STAGING, FRONTEND] + _ERROR_TARGETS

scope = TargetScope(
    mission_id=MID,
    allowed_targets=[Target(value=t, target_type=TargetType.CLOUD) for t in
                     (PLATFORM, STAGING, FRONTEND)],
    max_risk_level=RiskLevel.HIGH,
)
req = ContainerRequest(
    mission_id=MID, session_id="ses_phase12_container", scope=scope,
    mode=ContainerMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)

engine = app.container_engine
assert engine is not None and len(engine.capabilities) == 14
ids_seen = [c.capability_id for c in engine.capabilities]
assert set(ids_seen) == set(CONTAINER_CAPABILITY_IDS), (ids_seen, CONTAINER_CAPABILITY_IDS)
print("Registered container capabilities:", ", ".join(ids_seen))

for c in engine.capabilities:
    meta = c.meta()
    risk = meta.risk_level.value
    mode = meta.mode.value
    assert risk == "low", c.capability_id
    assert mode == "passive", c.capability_id
    assert meta.supported_target_types, c.capability_id
    assert set(t.value for t in meta.supported_target_types) <= {"asset", "cloud"}
    print(
        f"  {str(meta.id):<42} risk={risk:<7} mode={mode:<8} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )

# Multi-kind emissions: workload -> [workload, deployment], image -> [image,
# registry], resource_configuration -> [resource_configuration,
# configuration_discrepancy]; every other tool -> exactly one kind.
from blackforge.container import build_container_meta

_by_id = {str(m.id): [p.value for p in m.produces] for m in build_container_meta()}
assert _by_id["container.workload_observation"] == ["workload", "deployment"]
assert _by_id["container.image_metadata_observation"] == ["image", "registry"]
assert _by_id["container.resource_configuration_observation"] == [
    "resource_configuration", "configuration_discrepancy",
]
single_kind = {k: v for k, v in _by_id.items() if k not in {
    "container.workload_observation", "container.image_metadata_observation",
    "container.resource_configuration_observation"}}
assert all(len(v) == 1 for v in single_kind.values()), single_kind
CAPS_OK = True
print("Multi-kind produces (workload/image/resource_configuration) + single-kind others: PASS")

---

In [ ]:
from blackforge.container import (
    ContainerStatus,
    observation_confidence,
)
from blackforge.evidence.models import EvidenceRelation, EvidenceStatus, EvidenceType
from blackforge.core.types import Confidence
from blackforge.world_model.query import RelationshipQuery, WorldQuery
from blackforge.world_model.models import EntityType, RelationshipType, WorldLifecycle

TOOLS = [
    "observe_clusters", "observe_nodes", "enumerate_namespaces",
    "observe_workloads", "observe_pods", "observe_containers",
    "observe_image_metadata", "observe_services", "observe_ingress",
    "observe_rbac", "observe_service_accounts", "observe_network_policies",
    "observe_security_contexts", "observe_resource_configuration",
]
EXPECTED_COUNTS: dict[str, dict[str, int]] = {
    PLATFORM: {
        "observe_clusters": 1, "observe_nodes": 2, "enumerate_namespaces": 3,
        "observe_workloads": 6, "observe_pods": 4, "observe_containers": 4,
        "observe_image_metadata": 4, "observe_services": 2, "observe_ingress": 1,
        "observe_rbac": 2, "observe_service_accounts": 3,
        "observe_network_policies": 2, "observe_security_contexts": 3,
        "observe_resource_configuration": 4,
    },
    STAGING: {
        "observe_clusters": 1, "observe_nodes": 1, "enumerate_namespaces": 1,
        "observe_workloads": 2, "observe_pods": 1, "observe_containers": 1,
        "observe_image_metadata": 2, "observe_services": 1, "observe_ingress": 0,
        "observe_rbac": 1, "observe_service_accounts": 1,
        "observe_network_policies": 0, "observe_security_contexts": 1,
        "observe_resource_configuration": 1,
    },
    FRONTEND: {
        "observe_clusters": 1, "observe_nodes": 2, "enumerate_namespaces": 3,
        "observe_workloads": 2, "observe_pods": 2, "observe_containers": 2,
        "observe_image_metadata": 1, "observe_services": 1, "observe_ingress": 1,
        "observe_rbac": 1, "observe_service_accounts": 1,
        "observe_network_policies": 1, "observe_security_contexts": 1,
        "observe_resource_configuration": 3,
    },
}

_default_status = {
    (STAGING, "observe_ingress"): ContainerStatus.NO_EVIDENCE,
    (STAGING, "observe_network_policies"): ContainerStatus.NO_EVIDENCE,
}


def _pipeline_all(engine, req):
    runs = {}
    for target in (PLATFORM, STAGING, FRONTEND):
        for tool in TOOLS:
            result = getattr(engine, tool)(req, target)
            assert result.authorized is True, (target, tool)
            expected_status = _default_status.get((target, tool), ContainerStatus.SUCCESS)
            assert result.status == expected_status, (target, tool, result.status)
            assert result.observation_count == EXPECTED_COUNTS[target][tool], (
                target, tool, result.observation_count,
            )
            runs[(target, tool)] = result
    return runs


runs = _pipeline_all(engine, req)
statuses = []
for target in (PLATFORM, STAGING, FRONTEND):
    for tool in TOOLS:
        r = runs[(target, tool)]
        statuses.append(f"{target.split('/')[0]}.{tool}={r.status.value}({r.observation_count})")
print("All 14 container capabilities executed on all 3 targets")
print("Statuses:", " ".join(statuses))
obs = sum(r.observation_count for r in runs.values())
print(f"Total observations asserted: {obs}")

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
for r in runs.values():
    artifact = r.evidence_ids[0]
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 50, count_obs
print(f"DERIVED_FROM links: {count_obs} observations across {len(runs)} artifacts")

# Every row persists as OBSERVED (container evidence never elevates).
stored = {e.id: e.status for e in app.evidence_store.list(limit=10000)}
assert all(stored[ev] == EvidenceStatus.OBSERVED for r in runs.values()
           for ev in r.evidence_ids), "container evidence must stay OBSERVED"
print("All container evidence rows persisted with status OBSERVED")

total_evidence = app.evidence_store.count(MID)
assert total_evidence > 50, total_evidence
print(f"Evidence rows for mission: {total_evidence}")

# --- world materialization --------------------------------------------------
wm = app.world_model
entities = wm.list_entities(WorldQuery(mission_id=MID, limit=1000))
etypes = {e.entity_type.value for e in entities}
needed = {
    "cluster", "node", "namespace", "workload", "deployment", "pod",
    "container", "container_image", "registry", "service", "ingress",
    "role", "permission", "service_account", "network_policy",
}
assert needed <= etypes, etypes
print("World entity types:", ", ".join(sorted(etypes)))

rels = wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
rel_types = {r.relationship_type.value for r in rels}
need_rel = {
    "contains", "runs", "deploys", "uses_service_account", "belongs_to",
    "uses_image", "applies_to", "has_permission", "has_role", "selects",
    "routes_to",
}
assert need_rel <= rel_types, rel_types
offensive = {
    "exploits", "can_compromise", "leads_to", "enables",
    "privilege_escalation_path",
}
assert rel_types & offensive == set(), rel_types & offensive
print("Relationship types:", ", ".join(sorted(rel_types)))
print("No attack-graph relationship types (EXPLOITS/CAN_COMPROMISE/LEADS_TO/ENABLES): PASS")

# Headline totals across the whole mission (platform + staging + frontend).
_headline = {
    EntityType.CLUSTER: 2,
    EntityType.NODE: 3,
    EntityType.NAMESPACE: 4,
    EntityType.WORKLOAD: 4,
    EntityType.DEPLOYMENT: 4,
    EntityType.POD: 5,
    EntityType.CONTAINER: 4,
    EntityType.CONTAINER_IMAGE: 4,
    EntityType.REGISTRY: 2,
    EntityType.SERVICE: 3,
    EntityType.INGRESS: 1,
    EntityType.ROLE: 3,
    EntityType.PERMISSION: 3,
    EntityType.SERVICE_ACCOUNT: 4,
    EntityType.NETWORK_POLICY: 2,
}
for entity_type, expected_count in _headline.items():
    actual = wm.count_entities(MID, entity_type=entity_type, lifecycle=WorldLifecycle.ACTIVE)
    assert actual == expected_count, (entity_type, actual, expected_count)
    print(f"  {entity_type.value:<20} active={actual}")
assert wm.count_entities(MID, lifecycle=WorldLifecycle.ACTIVE) == 48

# Confidence policy: direct authoritative -> HIGH, derived -> MEDIUM, passive -> LOW.
cluster_obs = runs[(PLATFORM, "observe_clusters")].observations[0]
assert observation_confidence(cluster_obs, ContainerMode.CONTROLLED) == Confidence.HIGH
ingress_obs = runs[(PLATFORM, "observe_ingress")].observations[0]
assert observation_confidence(ingress_obs, ContainerMode.CONTROLLED) == Confidence.MEDIUM
assert observation_confidence(ingress_obs, ContainerMode.PASSIVE) == Confidence.LOW
print("Confidence policy (direct -> HIGH, derived -> MEDIUM, passive -> LOW): PASS")

# Security context assertions land on the container entity.
containers = wm.list_entities(
    WorldQuery(mission_id=MID, entity_type=EntityType.CONTAINER, limit=10))
assert containers
container_assertions = wm.list_assertions(
    str(containers[0].id), lifecycle=WorldLifecycle.ACTIVE)
keys = {a.property_key for a in container_assertions}
assert {"privileged", "allow_privilege_escalation", "run_as_non_root"}.issubset(keys)
print("Security-context assertions on containers (privileged / escalation / non-root): PASS")

# Resource configuration + discrepancy assertions on the web-api workload.
workloads = wm.list_entities(
    WorldQuery(mission_id=MID, entity_type=EntityType.WORKLOAD, limit=10))
web = next(w for w in workloads if w.name == "web-api")
web_keys = {a.property_key for a in wm.list_assertions(
    str(web.id), lifecycle=WorldLifecycle.ACTIVE)}
assert {"cpu_request", "cpu_limit", "memory_request", "memory_limit"}.issubset(web_keys)
discrepancies = [k for k in web_keys if k.startswith("discrepancy.")]
assert discrepancies, web_keys
disc_assertions = [a for a in wm.list_assertions(
    str(web.id), lifecycle=WorldLifecycle.ACTIVE)
    if a.property_key.startswith("discrepancy.")]
assert all(a.epistemic_status == EvidenceStatus.INFERRED for a in disc_assertions)
print("Resource-limit assertions + INFERRED configuration discrepancy on web-api: PASS")

---

In [ ]:
# Mission isolation: container work under a second mission is disjoint.
MID2 = "mission_phase12_container_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[Target(value=PLATFORM, target_type=TargetType.CLOUD)],
    max_risk_level=RiskLevel.HIGH,
)
req2 = ContainerRequest(
    mission_id=MID2, session_id="ses_phase12_container_2", scope=scope2,
    mode=ContainerMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)
r2 = engine.observe_pods(req2, PLATFORM)
other_ids = {str(x) for x in r2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in runs[(PLATFORM, "observe_pods")].evidence_ids})
assert app.evidence_store.count(MID2) == len(r2.evidence_ids)
assert wm.count_entities(MID2, entity_type=EntityType.POD,
                         lifecycle=WorldLifecycle.ACTIVE) == 4
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Namespaced target narrows rows without breaking the pipeline.
front = engine.observe_pods(req, FRONTEND)
assert front.observation_count == 2
assert all(o.namespace == "frontend" for o in front.observations)
print("Namespaced target aelionix-platform/frontend filtered to frontend pods: PASS")

# Redaction: the workloads artifact preserves structure but strips credentials.
rows = {e.id: e for e in app.evidence_store.list(limit=10000)}
artifact = rows[runs[(PLATFORM, "observe_workloads")].evidence_ids[0]]
assert artifact.evidence_type == EvidenceType.ARTIFACT
assert "demo-registry-token-" not in artifact.raw_data
assert "demo-sa-token-" not in artifact.raw_data
print("Workloads artifact redacted (no registry/service-account tokens stored): PASS")

from blackforge.container import redact_container_raw
import json

demo_raw = json.dumps({
    "cluster": PLATFORM,
    "namespace": "frontend",
    "workspace": "web-api",
    "registry_token": "demo-registry-token-0000",
    "service_account_token": "demo-sa-token-0000",
    "kubeconfig_password": "demo-kubeconfig-password-0000",
    "labels": ["api", "public"],
})
clean_raw = redact_container_raw(demo_raw)
clean = json.loads(clean_raw)
assert clean["registry_token"] == "REDACTED"
assert clean["service_account_token"] == "REDACTED"
assert clean["kubeconfig_password"] == "REDACTED"
assert clean["labels"] == ["api", "public"]
assert "demo-" not in clean_raw
print("Redaction unit behavior (credential-like fields -> stable REDACTED marker): PASS")

# Idempotency: a repeated full-pipeline run reuses rows.
before = app.evidence_store.count(MID)
runs2 = _pipeline_all(engine, req)
after = app.evidence_store.count(MID)
assert after == before, (before, after)
print()

---

In [ ]:
from blackforge.core.errors import AuthorizationError, ContainerExecutionError

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = True
try:
    engine.observe_pods(req, "ocs/sneaky-cluster")
    denied_out = False
except AuthorizationError:
    pass
assert denied_out, "out-of-scope target ocs/sneaky-cluster must be denied"
print("Out-of-scope target denied before transport execution: PASS")

# 2) Non-container target type is rejected (no generic execution surface).
unsupported_type = True
try:
    engine.observe_clusters(req, "apps.aelionix.test:8080")
    unsupported_type = False
except ContainerExecutionError:
    pass
assert unsupported_type, "non-container target type must be rejected"
print("Non-container target type rejected (no generic execution surface): PASS")

# 3) Unknown capability is rejected.
unknown_rejected = True
try:
    engine.run(req, "container.does_not_exist", PLATFORM)
    unknown_rejected = False
except ContainerExecutionError:
    pass
assert unknown_rejected, "unknown capability must be rejected"
print("Unknown capability rejected (no generic execution surface): PASS")

# 4) Invalid mode is rejected.
invalid_mode = True
try:
    engine.observe_pods(req, PLATFORM, mode="turbo")
    invalid_mode = False
except ContainerExecutionError:
    pass
assert invalid_mode, "invalid mode must be rejected"
print("Invalid mode rejected: PASS")

# 5) Failure states on the mock synthetic error targets.
_error_map = {
    "snail-cluster": ContainerStatus.TIMEOUT,
    "bursty-cluster": ContainerStatus.RATE_LIMITED,
    "locked-cluster": ContainerStatus.UNAUTHORIZED,
    "garbled-cluster": ContainerStatus.MALFORMED_RESPONSE,
    "fabricated-cluster": ContainerStatus.UNSUPPORTED_CLUSTER,
}
scope_err = TargetScope(
    mission_id=MID,
    allowed_targets=[
        Target(value=t, target_type=TargetType.ASSET)
        for t in tuple(_error_map) + ("ghost-cluster",)
    ],
    max_risk_level=RiskLevel.HIGH,
)
req_err = ContainerRequest(
    mission_id=MID, session_id="ses_phase12_container_err", scope=scope_err,
    mode=ContainerMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)
for target, expected in _error_map.items():
    got = engine.observe_clusters(req_err, target)
    assert got.status == expected, (target, got.status, expected)
print("Failure state mapping (5 synthetic error targets) verified: PASS")

# 6) Unknown cluster in scope fails closed with no observations.
ghost = engine.observe_clusters(req_err, "ghost-cluster")
assert ghost.status == ContainerStatus.UNKNOWN_CLUSTER
assert len(ghost.observations) == 0
print("Unknown cluster fails closed (UNKNOWN_CLUSTER, 0 observations): PASS")

# 7) Observation limit truncates instead of overflowing.
from blackforge.container import ContainerRequest as _CR

req_lim = _CR(
    mission_id=MID, session_id="ses_phase12_container_lim", scope=scope,
    mode=ContainerMode.CONTROLLED, max_observations=2, timeout_seconds=30.0,
)
limited = engine.observe_workloads(req_lim, PLATFORM)
assert limited.status == ContainerStatus.LIMITED
assert limited.observation_count == 2
assert len(limited.warnings) == 1
print("Observation limit truncates (LIMITED, 2 observations, warning): PASS")

# 8) Passive mode is LOW confidence and never collides with controlled records.
req_pas = _CR(
    mission_id=MID, session_id="ses_phase12_container_pas", scope=scope,
    mode=ContainerMode.PASSIVE, max_observations=500, timeout_seconds=30.0,
)
pas = engine.observe_clusters(req_pas, PLATFORM)
assert pas.mode == ContainerMode.PASSIVE
assert observation_confidence(pas.observations[0], pas.mode) == Confidence.LOW
pas_ids = {str(x) for x in pas.evidence_ids}
assert pas_ids.isdisjoint({str(x) for x in runs[(PLATFORM, "observe_clusters")].evidence_ids})
print("Confidence mode policy (PASSIVE -> LOW, separate evidence rows): PASS")

---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == wm.count_entities(MID)
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert persisted_ev and persisted_wm and rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "container" / "engine.py").exists(),
    "phase12_modules": bool(
        (REPO_DIR / "blackforge" / "container" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "redaction.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "normalization.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "transport.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "addressing.py").exists()
        and (REPO_DIR / "blackforge" / "container" / "canonical.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_container_ready": BOOTSTRAP_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "evidence_observed": bool(count_obs >= 50),
    "world_materialized": bool(needed <= etypes),
    "no_attack_graph": not bool(rel_types & offensive),
    "confidence_policy": True,
    "scope_authorization": denied_out,
    "unsupported_type_rejected": unsupported_type,
    "unknown_capability_rejected": unknown_rejected,
    "invalid_mode_rejected": invalid_mode,
    "redaction_boundary": True,
    "namespaced_target": bool(all(o.namespace == "frontend" for o in front.observations)),
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in runs[(PLATFORM, "observe_pods")].evidence_ids})),
    "idempotent_runs": bool(after == before),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_container_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["scope_authorization"]
    and phase_checks["unsupported_type_rejected"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["invalid_mode_rejected"]
    and phase_checks["redaction_boundary"]
    and phase_checks["no_attack_graph"]
    and phase_checks["idempotent_runs"]
)

print()
print("=" * 60)
print("PHASE 12 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---